<div style="border-radius: 10px; padding: 32px 0px; border: 1px solid rgba(128,128,128,0.2);">
  <div style="display: flex; justify-content: space-between; align-items: flex-start; flex-wrap: wrap; gap: 16px; padding: 0px 32px;">
  <div>
    <div style="font-size: 0.75rem; letter-spacing: 3px; text-transform: uppercase; font-weight: 600; margin-bottom: 10px; opacity: 0.6;">
      Máster Universitario en Big Data y Computación en la Nube.
    </div>
    <div style="font-size: 1.5rem; font-weight: 700; margin-bottom: 4px;">Trabajo de Fin de Máster</div>
    <div style="font-size: 1rem; font-weight: 400; opacity: 0.75;">Clasificador taxonómico de boletines oficiales españoles</div>
  </div>
  <div style="margin-top: 16px; display: flex; align-items: center; gap: 12px;">
    <div style="font-size: 1rem; font-weight: 600;">Hugo de Lamo</div>
  </div>
  </div>
</div>

# 05 · Clasificador taxonómico de boletines oficiales

Este notebook implementa un clasificador multietiqueta de publicaciones de boletines oficiales españoles usando **Pydantic AI**. El problema es una aguja en un pajar: de ~65 000 publicaciones del Q1 2025, solo ~3,7 % son relevantes para el dominio ambiental-energético.

El clasificador responde cuatro preguntas por publicación:
1. **¿Es relevante?**  ¿Pertenece al universo de autorizaciones ambiental-energéticas?
2. **¿Qué procedimientos contiene?**  Lista multilabel: DIA, AAP, AAC, AAU, IIA, AAI, IAE, DUP.
3. **¿Qué tipo de acto es?**  Forma jurídica del documento (N1): resolución, anuncio, decreto…
4. **¿Qué tecnología menciona?** Lista multilabel: fotovoltaica, eólica, hidrógeno…

---

## Estructura del notebook

### Parte I - Fundamentos
|   | Sección | Contenido |
|---|---------|----------|
| **0** | **Setup** | Entorno, dependencias, modelo local |
| **1** | **Schema de output** | `ClassifierOutput`, enums e invariantes |
| **2** | **Pre-procesamiento** | N0 lookup + N1 clasificador por reglas |
| **3** | **Ground truth** | Muestreo estratificado + anotación manual |
| **4** | **Agente base** | System prompt, construcción y casos cualitativos |

### Parte II - Ciclo de experimentación
|   | Sección | Contenido |
|---|---------|----------|
| **5** | **Experimento 1 - Baseline** | Zero-shot, sin contexto N1 |
| **6** | **Análisis de errores** | Qué falla y por qué |
| **7** | **Prompt v2** | Mejora basada en errores (DEC-013 a DEC-019) |
| **8** | **Experimento 2 - Prompt v2** | ¿Mejora respecto a baseline? |
| **9** | **Experimento 3 - Ablación +N1** | ¿Aporta el contexto de forma? |
| **10** | **Experimento 4 - Few-shot** | ¿Ayudan los ejemplos reales? |
| **11** | **Comparativa de modelos** | Qwen 3.5 9B vs Gemma 4 4B |

### Parte III - Análisis final
|   | Sección | Contenido |
|---|---------|----------|
| **12** | **Tabla resumen** | Comparativa de todos los experimentos |
| **13** | **Calibración de confianza** | ¿El modelo sabe cuándo no sabe? |
| **14** | **Conclusiones y trabajo futuro** | Hallazgos, limitaciones, v2 |

---

##  0. Setup

Cargamos las variables de entorno e importamos las librerías. El modelo vive en LM Studio - el único punto de cambio para conectar otro proveedor es `LM_STUDIO_MODEL`.

In [ ]:
# Librerías estándar
import os
import html
import re
import json
import asyncio
from pathlib import Path

# Datos
import pandas as pd

# Carga de variables de entorno desde .env (busca hacia arriba desde el directorio actual)
from dotenv import find_dotenv, load_dotenv

# Pydantic AI - framework que fuerza al LLM a devolver JSON validado por Pydantic
from pydantic_ai import Agent

load_dotenv(find_dotenv())


In [ ]:
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

# Cambiar LM_STUDIO_MODEL según el modelo cargado en LM Studio
LM_STUDIO_MODEL = "qwen/qwen3.5-9b"

model = OpenAIModel(
    LM_STUDIO_MODEL,
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)


In [ ]:
import httpx

try:
    r = httpx.get("http://localhost:1234/v1/models", timeout=3)
    modelos = [m["id"] for m in r.json()["data"]]
    assert LM_STUDIO_MODEL in modelos, f"{LM_STUDIO_MODEL} no está cargado en LM Studio"
    print(f"✓ {LM_STUDIO_MODEL} listo  |  otros cargados: {[m for m in modelos if m != LM_STUDIO_MODEL]}")
except httpx.ConnectError:
    raise RuntimeError("LM Studio no responde - ¿está el servidor arrancado?")


---

##  1. Schema de output

El schema define el **contrato entre el LLM y el sistema**: qué campos devuelve el modelo, de qué tipo y bajo qué restricciones. Pydantic valida cada respuesta antes de que llegue al resto del código, forzando un retry automático si algo no cumple el schema.

`ClassifierOutput` tiene 6 campos:

| Campo | Tipo | Rol |
|-------|------|-----|
| `is_relevant` | `bool` | ¿Pertenece al dominio ambiental-energético? |
| `act_type` | `ActType` | Forma jurídica del acto (N1) - resolución, anuncio, decreto… |
| `procedures` | `list[ProcedureType]` | Procedimientos identificados (N2) - multilabel |
| `technologies` | `list[TechnologyType]` | Tecnologías mencionadas (N3) - multilabel, puede ser vacía |
| `confidence` | `float` | Confianza global entre 0.0 y 1.0 |
| `reasoning` | `str` | Justificación breve citando el texto que dispara cada etiqueta |

Un `@model_validator` impone los invariantes de negocio: `is_relevant=True` exige `procedures != []`; `is_relevant=False` exige ambas listas vacías.

In [ ]:
# Importar los tipos del schema: enums de etiquetas y el modelo de output del clasificador
from clasificador.schema import ActType, ProcedureType, TechnologyType, ClassifierOutput


In [13]:
# Verificación de invariantes
ejemplo_valido = ClassifierOutput(
    is_relevant=True,
    act_type=ActType.RESOLUCION,
    procedures=[ProcedureType.DIA],
    technologies=[TechnologyType.FOTOVOLTAICA],
    confidence=0.98,
    reasoning="'se formula la declaración de impacto ambiental' → DIA. 'Planta Solar Fotovoltaica' → fotovoltaica.",
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

print("\nViolación de invariante:")
try:
    ClassifierOutput(
        is_relevant=False, act_type=ActType.RESOLUCION,
        procedures=[ProcedureType.AAP], technologies=[],
        confidence=0.5, reasoning="Prueba.",
    )
except Exception as e:
    print(f"  ValidationError → {e.errors()[0]['msg']}")

Ejemplo válido:
{
  "is_relevant": true,
  "act_type": "resolución",
  "procedures": [
    "DIA"
  ],
  "technologies": [
    "fotovoltaica"
  ],
  "confidence": 0.98,
  "reasoning": "'se formula la declaración de impacto ambiental' → DIA. 'Planta Solar Fotovoltaica' → fotovoltaica."
}

Violación de invariante:
  ValidationError → Value error, is_relevant=False con procedures != []


---

##  2. Pre-procesamiento

Antes de llamar al LLM, cada registro pasa por dos pasos deterministas implementados en `src/clasificador/agent.py`:

- **N0 - Ámbito**: lookup directo del campo `bulletin` → `estatal / autonómico / local`. Sin LLM.
- **N1 - Tipo de acto**: clasificador de primer token con pre-procesamiento de formatos especiales (BOCM, BOCA, BOIB, BOE topónimos).

**Cobertura medida**: 89.6% del corpus (Q1 2025) tras tres iteraciones de mejora.

In [ ]:
# Las funciones N0/N1 viven en src/clasificador/agent.py para poder importarlas
# tanto desde notebooks como desde scripts de producción sin duplicar código
from clasificador.agent import get_ambito, inferir_act_type, preprocess_description

PATH_PARQUET = "../data/raw/silver_official_gazettes_2025_Q1.parquet"
df = pd.read_parquet(PATH_PARQUET)

# Las descripciones del scraping llevan entidades HTML (ej. &amp; → &, &lt; → <)
df["description"] = df["description"].apply(html.unescape)

print(f"Corpus: {len(df):,} registros · {df['bulletin'].nunique()} boletines")


In [ ]:
# Aplicar el clasificador N1 (reglas de primer token) a todo el corpus
# y auditar la cobertura antes de pasar al LLM
df["act_type_n1"] = df.apply(
    lambda row: inferir_act_type(row["description"], row["bulletin"]), axis=1
)

dist = df["act_type_n1"].value_counts()
total = len(df)
print("Distribución N1 inferida:\n")
for val, count in dist.items():
    print(f"  {val:<25} {count:>6,}  ({count/total*100:.1f}%)")

# OTROS agrupa todo lo que no encaja en ningún patrón - mide el hueco de cobertura
otros = (df["act_type_n1"] == ActType.OTROS).sum()
print(f"\nCobertura N1: {(1 - otros/total)*100:.1f}%  ({otros:,} en OTROS)")


### Límites del clasificador N1 y trabajo futuro

El clasificador de primer token cubre **89.6%** del corpus con reglas deterministas.
El 10.4% restante cae en `OTROS` por tres motivos con soluciones conocidas:

| Grupo | Volumen aprox. | Motivo | Solución futura |
|---|---|---|---|
| Topónimos BOE / subastas AEAT | ~5.200 | Sin contenido textual real - irrecuperable sin PDF | Clase propia `NO_INFERIBLE` en v2 |
| BON fiscal | ~160 | `tipos`, `calendario` - formatos tributarios navarros | Reglas específicas BON |
| RRHH sin tipo explícito | ~350 | `relación`, `lista`, `oferta`, `bajas` - tipo de acto no en la descripción | LLM zero-shot viable en v2 (DEC-011) |
| BOIB residual | ~300 | Variantes de prefijo catalán no contempladas | Ampliar split `.-` para BOIB |
| Long tail distribuido | ~740 | Tokens poco frecuentes sin patrón claro | Cobertura ~98.5% teórica con PDF |

**Techo práctico estimado sin PDF**: ~98.5% añadiendo más tokens al mapa.
**Techo real con PDF**: ~100% - el tipo de acto aparece siempre en el cuerpo del documento.

---

##  3. Ground truth

El ground truth se construye en tres capas:

| Capa | Qué etiqueta | Cómo | 
|------|-------------|------|
| **1 - N1 determinista** | `act_type` | Reglas de primer token ( 2) | 
| **2 - Muestreo estratificado** | Selección de 100 registros | Keywords por procedimiento N2 | 
| **3 - Anotación manual** | `is_relevant_gt`, `procedures_gt`, `technologies_gt` | Revisión humana |

El archivo `ground_truth_100_anotado.csv` contiene los 100 registros con etiquetas manuales validadas.

In [9]:
import random
random.seed(42)

keywords = {
    "DIA":     ["declaración de impacto ambiental"],
    "AAP":     ["autorización administrativa previa"],
    "AAC":     ["autorización de construcción", "autorización administrativa de construcción"],
    "AAP_AAC": ["previa y de construcción"],
    "AAU":     ["autorización ambiental unificada"],
    "IIA":     ["informe de impacto ambiental"],
    "AAI":     ["autorización ambiental integrada"],
    "IAE":     ["ambiental estratégic"],
    "DUP":     ["utilidad pública"],
}
cuotas = {"DIA": 60, "AAP": 60, "AAC": 50, "AAP_AAC": 40,
          "AAU": 40, "IIA": 40, "AAI": 30, "IAE": 30, "DUP": 30}
N_NEGATIVOS = 120

sampled_ids = set()
frames = []

for grupo, kws in keywords.items():
    mask = df["description"].str.lower().str.contains("|".join(kws), na=False)
    mask = mask & ~df.index.isin(sampled_ids)
    pool = df[mask]
    n = min(cuotas[grupo], len(pool))
    sample = pool.sample(n, random_state=42).copy()
    sample["grupo_muestreo"] = grupo
    sampled_ids.update(sample.index.tolist())
    frames.append(sample)
    print(f"  {grupo:<10} pool={len(pool):>5,}  sampled={n}")

all_kws = [kw for kws in keywords.values() for kw in kws]
mask_neg = ~df["description"].str.lower().str.contains("|".join(all_kws), na=False)
mask_neg = mask_neg & ~df.index.isin(sampled_ids)
negativos = df[mask_neg].sample(N_NEGATIVOS, random_state=42).copy()
negativos["grupo_muestreo"] = "NEGATIVO"
frames.append(negativos)

df_gt_full = pd.concat(frames, ignore_index=True)
df_gt_full["id"] = range(len(df_gt_full))
print(f"\nTotal muestreado: {len(df_gt_full)} registros")

  DIA        pool=  286  sampled=60
  AAP        pool=1,016  sampled=60
  AAC        pool=  514  sampled=50
  AAP_AAC    pool=  243  sampled=40
  AAU        pool=  182  sampled=40
  IIA        pool=  521  sampled=40
  AAI        pool=  257  sampled=30
  IAE        pool=  319  sampled=30
  DUP        pool=  635  sampled=30

Total muestreado: 500 registros


### Dataset anotado

Los 100 registros del ground truth han sido anotados manualmente. Durante el proceso se identificaron casos especiales documentados en `decisiones_implementacion.md` (DEC-012 a DEC-019):

- **Denegaciones**: heredan el tipo de procedimiento del acto denegado
- **Modificaciones**: heredan los procedimientos del acto modificado
- **Falsos positivos**: RRHH con vocabulario ambiental, concesiones de dominio público no energéticas
- **Tecnologías emergentes v2**: `industria_ippc`, `infraestructura_hidrica`, `ordenacion_territorial`, `turismo_edificacion`

In [ ]:
# Cargar el CSV anotado manualmente (100 registros con etiquetas ground truth)
PATH_GT_ANOTADO = "../data/ground_truth/ground_truth_100_anotado.csv"
df_anotado = pd.read_csv(PATH_GT_ANOTADO)

print(f"Ground truth: {len(df_anotado)} registros")
print(f"Relevantes: {df_anotado['is_relevant_gt'].sum()} | No relevantes: {(~df_anotado['is_relevant_gt'].astype(bool)).sum()}")
print()
print("Distribución N2:")
print(df_anotado["procedures_gt"].value_counts().to_string())


---

##  4. Agente base

Toda la lógica de clasificación vive en `src/clasificador/`. El notebook solo importa y orquesta.

- `schema.py` → `ClassifierOutput` y enums
- `prompts.py` → `SYSTEM_PROMPT_V1/V2/V3`
- `agent.py` → `build_agent`, `run_experiment`, pre-procesamiento N0/N1

In [ ]:
# Importar la capa de agente desde el módulo clasificador:
# - build_agent:      construye un Agent con el modelo y prompt indicados
# - run_experiment:   ejecuta el batch completo con semáforo y barra de progreso
# - clasificar_async: clasifica un único registro de forma asíncrona
from clasificador.agent import build_agent, run_experiment, clasificar_async
from clasificador.prompts import SYSTEM_PROMPT_V1, SYSTEM_PROMPT_V2, SYSTEM_PROMPT_V3, PROMPT_REGISTRY
from tqdm.asyncio import tqdm_asyncio


In [ ]:
# Construir el agente con la mejor configuración conocida (Prompt v3 - few-shot)
# para usarlo en los casos cualitativos de validación y en el Experimento 4
agent_v3 = build_agent(model, prompt_version="v3")


In [ ]:
# Validación cualitativa - 5 casos antes del batch
casos = [
    ("Resolución de 12 de marzo de 2025, de la Dirección General de Calidad y Evaluación "
     "Ambiental, por la que se formula la declaración de impacto ambiental del proyecto "
     "Planta Solar Fotovoltaica Los Llanos, en la provincia de Cáceres.", "doe"),
    ("Resolución de 5 de febrero de 2025, de la Dirección General de Política Energética, "
     "por la que se otorga autorización administrativa previa y de construcción para el "
     "Parque Eólico Sierra Norte, de 48 MW, en Salamanca.", "boe"),
    ("Resolución de 18 de enero de 2025, de la Delegación Territorial de Medio Ambiente, "
     "por la que se otorga autorización ambiental unificada para la planta de biogás "
     "Valdecorneja, en Ávila.", "boja"),
    ("Resolución de 3 de marzo de 2025, de la Universidad de Salamanca, por la que se "
     "convoca concurso-oposición para cubrir plazas de profesor ayudante doctor.", "bocyl"),
    ("Resolución de 21 de febrero de 2025, de la Dirección General de Medio Natural, "
     "por la que se formula el informe de impacto ambiental del proyecto de línea "
     "eléctrica subterránea de 132 kV en Zaragoza.", "boa"),
]
EXPECTED = [
    {"procedures": {"DIA"}, "relevant": True},
    {"procedures": {"AAP","AAC"}, "relevant": True},
    {"procedures": {"AAU"}, "relevant": True},
    {"procedures": set(), "relevant": False},
    {"procedures": {"IIA"}, "relevant": True},
]

print("Validación cualitativa - 5 casos\n")
aciertos = 0
for i, (desc, bul) in enumerate(casos, 1):
    result = await agent_v3.run(
        f"Boletín: {bul.upper()} (ámbito: {get_ambito(bul)})\n\nDescripción: {desc}"
    )
    r = result.output
    pred_proc = set(p.value for p in r.procedures)
    exp = EXPECTED[i-1]
    ok = (r.is_relevant == exp["relevant"] and pred_proc == exp["procedures"])
    aciertos += ok
    print(f"{'✅' if ok else '❌'} Caso {i} | is_relevant={r.is_relevant} | procedures={pred_proc}")
print(f"\nResultado: {aciertos}/5 correctos")


---

##  5. Experimento 1 - Baseline

Zero-shot con `SYSTEM_PROMPT_V1`. Sin contexto N1.

**Modelo**: Qwen 3.5 9B · **Concurrencia**: 1 (DEC-009: modelos locales secuenciales)

In [ ]:
# Construir el agente con el prompt baseline (v1)
agent_v1 = build_agent(model, prompt_version="v1")

# Ejecutar Experimento 1 - requiere LM Studio activo con el modelo cargado
# Si se interrumpe a mitad, relanzar la celda retomará desde el último registro guardado
df_exp1 = await run_experiment(
    agent_v1, df_anotado,
    use_n1_context=False, concurrency=1,
    output_path="../results/exp1_baseline_qwen9b.csv",
    desc="Exp1 · Baseline",
)


---

##  6. Análisis de errores - Baseline

Evaluamos las predicciones del Exp 1 contra el ground truth para identificar patrones de error que guíen la mejora del prompt en §7.

In [ ]:
# Convierte cualquier representación de etiquetas a un set de strings.
# Necesario porque el ground truth usa CSV ("DIA,AAP") y las predicciones
# del agente usan JSON ('["DIA","AAP"]') - esta función normaliza ambos formatos.
def parse_labels(value) -> set:
    if pd.isna(value) or str(value).strip() in ("", "nan"):
        return set()
    v = str(value).strip()
    # Intentar parsear como JSON primero (formato de las predicciones)
    if v.startswith("["):
        try:
            return set(json.loads(v))
        except Exception:
            pass
    # Fallback: split por comas (formato del ground truth manual)
    return set(x.strip() for x in v.split(",") if x.strip())


In [ ]:
def compute_metrics(df_eval, label=""):
    N2 = ["DIA","AAP","AAC","AAU","IIA","AAI","IAE","DUP"]
    y_true = df_eval["is_relevant_gt"].astype(bool)
    y_pred = df_eval["is_relevant_pred"].fillna(False).astype(bool)

    # -- is_relevant: métricas binarias de detección de relevancia --
    tp=((y_true)&(y_pred)).sum(); fp=((~y_true)&(y_pred)).sum()
    fn=((y_true)&(~y_pred)).sum(); tn=((~y_true)&(~y_pred)).sum()
    p=tp/(tp+fp) if tp+fp>0 else 0; r=tp/(tp+fn) if tp+fn>0 else 0
    f1_rel=2*p*r/(p+r) if p+r>0 else 0
    if label: print(f"── {label} ──────────────────────────")
    print(f"is_relevant  P={p:.3f}  R={r:.3f}  F1={f1_rel:.3f}  TP={tp} FP={fp} FN={fn} TN={tn}\n")

    # -- N2: precisión, recall y F1 por cada etiqueta de procedimiento --
    print(f"{'Label':<8} {'P':>6} {'R':>6} {'F1':>6} {'Sup':>5} {'TP':>4} {'FP':>4} {'FN':>4}")
    print("─" * 55)
    macro = 0
    for lbl in N2:
        yt = df_eval.apply(lambda r: lbl in parse_labels(r["procedures_gt"]), axis=1)
        yp = df_eval.apply(lambda r: lbl in parse_labels(r["procedures_pred"]), axis=1)
        tp2=(yt&yp).sum(); fp2=(~yt&yp).sum(); fn2=(yt&~yp).sum()
        p2=tp2/(tp2+fp2) if tp2+fp2>0 else 0
        r2=tp2/(tp2+fn2) if tp2+fn2>0 else 0
        f12=2*p2*r2/(p2+r2) if p2+r2>0 else 0; macro+=f12
        print(f"{lbl:<8} {p2:>6.3f} {r2:>6.3f} {f12:>6.3f} {yt.sum():>5} {tp2:>4} {fp2:>4} {fn2:>4}")
    print("─" * 55)
    print(f"{'Macro-F1':<8} {macro/len(N2):>6.3f}")

    # -- Exact match: el set de procedimientos predicho coincide exactamente con el GT --
    exact = df_eval.apply(
        lambda r: parse_labels(r["procedures_gt"]) == parse_labels(r["procedures_pred"]), axis=1
    )
    rel = y_true
    print(f"\nExact match (relevantes): {exact[rel].mean():.3f}  ({exact[rel].sum()}/{rel.sum()})")
    print(f"Exact match (todos):      {exact.mean():.3f}  ({exact.sum()}/{len(df_eval)})")

    # -- Confianza: distribución del score autoreportado por el LLM --
    conf = df_eval["confidence"].dropna()
    print(f"Confianza: media={conf.mean():.3f}  min={conf.min():.3f}  max={conf.max():.3f}")
    return exact, rel


In [ ]:
# Muestra los registros relevantes donde la predicción N2 no coincide con el GT.
# Parámetros:
#   exact - serie bool devuelta por compute_metrics (True si el set N2 es correcto)
#   rel   - serie bool con is_relevant_gt=True
#   n     - límite de errores a mostrar (None = todos)
def print_errors(df_eval, exact, rel, label="", n=None):
    errores = df_eval[~exact & rel]
    if n is not None:
        errores = errores.head(n)
    print(f"── Errores N2 en relevantes{' · ' + label if label else ''} ──")
    print(f"Total: {(~exact & rel).sum()}\n")
    for _, row in errores.iterrows():
        gt   = parse_labels(row["procedures_gt"])
        pred = parse_labels(row["procedures_pred"])
        print(f"ID {row['id']} | GT={sorted(gt)} | PRED={sorted(pred)}")
        print(f"  Falta: {sorted(gt - pred)} | Sobra: {sorted(pred - gt)}")
        print(f"  {str(row['description'])[:100]}...")
        # El reasoning del LLM explica qué leyó en el texto para tomar cada decisión
        print(f"  Razonamiento: {str(row.get('reasoning', ''))[:150]}")
        print()


In [ ]:
df_exp1 = pd.read_csv("../results/exp1_baseline_qwen9b.csv")
df_eval1 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp1[["description","is_relevant_pred","act_type_pred","procedures_pred",
             "technologies_pred","confidence","reasoning"]], on="description", how="left"
)
exact1, rel1 = compute_metrics(df_eval1, "Experimento 1 - Baseline")


In [ ]:
print_errors(df_eval1, exact1, rel1, label="Experimento 1 - Baseline")


---

##  7. Prompt v2 - Mejora basada en errores

**Cambios clave vs V1**: scope amplio para is_relevant, regla DIA+AAI, refuerzo anuncios de IP y desistimientos.

In [ ]:
# Mostrar el prompt v2 para comparar con v1 antes de lanzar el experimento
print(SYSTEM_PROMPT_V2)


---

##  8. Experimento 2 - Prompt v2

In [ ]:
# Construir el agente v2 (mismo modelo, prompt mejorado)
agent_v2 = build_agent(model, prompt_version="v2")

# Ejecutar Experimento 2 - sin contexto N1 para aislar el efecto del prompt
df_exp2 = await run_experiment(
    agent_v2, df_anotado,
    use_n1_context=False, concurrency=1,
    output_path="../results/exp2_promptv2_qwen9b.csv",
    desc="Exp2 · Prompt v2",
)


In [ ]:
# Evaluar Experimento 2 - Prompt v2
df_exp2 = pd.read_csv("../results/exp2_promptv2_qwen9b.csv")
df_eval2 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp2[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)
exact2, rel2 = compute_metrics(df_eval2, "Experimento 2 - Prompt v2")


In [ ]:
print_errors(df_eval2, exact2, rel2, label="Experimento 2 - Prompt v2")


---

##  9. Experimento 3 - Ablación +N1

**Pregunta**: ¿cuánto aporta pasar el `act_type` pre-computado como contexto al LLM?

In [ ]:
df_exp3 = await run_experiment(
    agent_v2, df_anotado,
    use_n1_context=True,  # añade act_type N1 pre-computado al mensaje
    concurrency=1,
    output_path="../results/exp3_promptv2_n1_qwen9b.csv",
    desc="Exp3 · v2 + N1",
)


In [ ]:
# Evaluar Experimento 3 - Prompt v2 + contexto N1
df_exp3 = pd.read_csv("../results/exp3_promptv2_n1_qwen9b.csv")
df_eval3 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp3[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)
exact3, rel3 = compute_metrics(df_eval3, "Experimento 3 - Prompt v2 + N1")


In [ ]:
print_errors(df_eval3, exact3, rel3, label="Experimento 3 - Prompt v2 + N1")


---

##  10. Experimento 4 - Few-shot

3 ejemplos quirúrgicos sobre errores del Exp 2 (DEC-020). Definido en `SYSTEM_PROMPT_V3`.

In [ ]:
# agent_v3 ya está construido en §4, no hace falta redefinirlo
df_exp4 = await run_experiment(
    agent_v3, df_anotado,
    use_n1_context=False, concurrency=1,
    output_path="../results/exp4_fewshot_qwen9b.csv",
    desc="Exp4 · Few-shot",
)


In [ ]:
# Evaluar Experimento 4 - Few-shot (Prompt v3)
df_exp4 = pd.read_csv("../results/exp4_fewshot_qwen9b.csv")
df_eval4 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp4[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)
exact4, rel4 = compute_metrics(df_eval4, "Experimento 4 - Few-shot (Prompt v3)")


In [ ]:
print_errors(df_eval4, exact4, rel4, label="Experimento 4 - Few-shot")


---

##  11. Comparativa de modelos

Gemma 4 4B con Prompt v2. Para ejecutar: cargar `gemma-4-e4b-it` en LM Studio y reiniciar el kernel.

In [ ]:
# Cargar 'gemma-4-e4b-it' en LM Studio antes de ejecutar
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

model_gemma = OpenAIChatModel(
    "gemma-4-e4b-it",
    provider=OpenAIProvider(base_url="http://localhost:1234/v1", api_key="lm-studio"),
)
agent_gemma = build_agent(model_gemma, prompt_version="v2")

df_exp5 = await run_experiment(
    agent_gemma, df_anotado,
    use_n1_context=False, concurrency=1,
    output_path="../results/exp5_promptv2_gemma4b.csv",
    desc="Exp5 · Gemma 4B",
)


In [ ]:
# Evaluar Experimento 5 - Gemma 4 4B
df_exp5 = pd.read_csv("../results/exp5_promptv2_gemma4b.csv")
df_eval5 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp5[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)
exact5, rel5 = compute_metrics(df_eval5, "Experimento 5 - Gemma 4 4B · Prompt v2")


In [ ]:
print_errors(df_eval5, exact5, rel5, label="Experimento 5 - Gemma 4B")


---

##  12. Tabla resumen - Comparativa de experimentos

In [ ]:
import os

# Configuración de los 5 experimentos: nombre legible, modelo, config y ruta al CSV
experimentos_cfg = [
    ("Exp 1 - Baseline zero-shot",   "Qwen 3.5 9B", "Zero-shot",   "../results/exp1_baseline_qwen9b.csv"),
    ("Exp 2 - Prompt v2",            "Qwen 3.5 9B", "Zero-shot",   "../results/exp2_promptv2_qwen9b.csv"),
    ("Exp 3 - v2 + N1",              "Qwen 3.5 9B", "+N1 context", "../results/exp3_promptv2_n1_qwen9b.csv"),
    ("Exp 4 - Few-shot (v3)",        "Qwen 3.5 9B", "Few-shot",    "../results/exp4_fewshot_qwen9b.csv"),
    ("Exp 5 - Gemma 4B - Prompt v2", "Gemma 4 4B",  "Zero-shot",   "../results/exp5_promptv2_gemma4b.csv"),
]
N2 = ["DIA","AAP","AAC","AAU","IIA","AAI","IAE","DUP"]

rows = []
for nombre, modelo, config, path in experimentos_cfg:
    # Cargar predicciones y alinearlas con el ground truth por descripción
    df_r = pd.read_csv(path)
    df_e = df_anotado[["id","is_relevant_gt","procedures_gt","description"]].merge(
        df_r[["description","is_relevant_pred","procedures_pred","confidence","reasoning"]],
        on="description", how="left"
    )

    # Métricas is_relevant
    y_true=df_e["is_relevant_gt"].astype(bool); y_pred=df_e["is_relevant_pred"].fillna(False).astype(bool)
    tp=((y_true)&(y_pred)).sum(); fp=((~y_true)&(y_pred)).sum(); fn=((y_true)&(~y_pred)).sum()
    p=tp/(tp+fp) if tp+fp>0 else 0; r=tp/(tp+fn) if tp+fn>0 else 0
    f1_rel=2*p*r/(p+r) if p+r>0 else 0

    # Macro-F1: media del F1 de cada etiqueta N2 (trata todas las etiquetas igual)
    macro=0
    for lbl in N2:
        yt=df_e.apply(lambda r: lbl in parse_labels(r["procedures_gt"]), axis=1)
        yp=df_e.apply(lambda r: lbl in parse_labels(r["procedures_pred"]), axis=1)
        tp2=(yt&yp).sum(); fp2=(~yt&yp).sum(); fn2=(yt&~yp).sum()
        p2=tp2/(tp2+fp2) if tp2+fp2>0 else 0; r2=tp2/(tp2+fn2) if tp2+fn2>0 else 0
        macro+=2*p2*r2/(p2+r2) if p2+r2>0 else 0

    # Exact match: fracción de registros donde el set de N2 es exactamente igual al GT
    exact=df_e.apply(lambda r: parse_labels(r["procedures_gt"])==parse_labels(r["procedures_pred"]), axis=1)
    conf=df_e["confidence"].dropna()
    rows.append({
        "Experimento": nombre, "Modelo": modelo, "Config": config,
        "is_rel F1": f"{f1_rel:.3f}", "Macro-F1": f"{macro/len(N2):.3f}",
        "Exact(rel)": f"{exact[y_true].mean():.3f}", "Exact(all)": f"{exact.mean():.3f}",
        "Conf media": f"{conf.mean():.3f}", "Conf min": f"{conf.min():.3f}",
        "Errores fmt": df_e["reasoning"].astype(str).str.startswith("ERROR").sum(),
    })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))
df_summary.to_csv("../results/tabla_resumen_experimentos.csv", index=False)


---

##  13. Visualizaciones

Comparativa gráfica de los 5 experimentos: F1 global, F1 por etiqueta N2 y distribución de confianza.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Extraer etiquetas cortas y valores de la tabla resumen ya calculada
exp_labels = [r["Experimento"].split("-")[1].strip() if "-" in r["Experimento"] else r["Experimento"] for r in rows]
macro_vals = [float(r["Macro-F1"]) for r in rows]
isrel_vals = [float(r["is_rel F1"]) for r in rows]
x = range(len(rows))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, vals, title, color in [
    (axes[0], macro_vals, "Macro-F1  (N2 procedimientos)", "#4C72B0"),
    (axes[1], isrel_vals, "F1  is_relevant",               "#DD8452"),
]:
    bars = ax.bar(x, vals, color=color, width=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(exp_labels, rotation=18, ha="right", fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("F1")
    ax.set_title(title)
    ax.axhline(1.0, color="gray", linewidth=0.5, linestyle="--")
    # Anotar el valor numérico encima de cada barra
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("../results/fig_comparativa_f1.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np

N2 = ["DIA","AAP","AAC","AAU","IIA","AAI","IAE","DUP"]
heatmap_data = []
hm_labels    = []

for nombre, modelo, config, path in experimentos_cfg:
    df_r = pd.read_csv(path)
    df_e = df_anotado[["procedures_gt","description"]].merge(
        df_r[["description","procedures_pred"]], on="description", how="left"
    )
    row_f1 = []
    for lbl in N2:
        yt = df_e.apply(lambda r: lbl in parse_labels(r["procedures_gt"]),  axis=1)
        yp = df_e.apply(lambda r: lbl in parse_labels(r["procedures_pred"]), axis=1)
        tp=(yt&yp).sum(); fp=(~yt&yp).sum(); fn=(yt&~yp).sum()
        p=tp/(tp+fp) if tp+fp>0 else 0; r=tp/(tp+fn) if tp+fn>0 else 0
        row_f1.append(2*p*r/(p+r) if p+r>0 else 0)
    heatmap_data.append(row_f1)
    hm_labels.append(nombre.split("-")[1].strip() if "-" in nombre else nombre)

matrix = np.array(heatmap_data)
fig, ax = plt.subplots(figsize=(10, len(hm_labels) * 0.8 + 1))
im = ax.imshow(matrix, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(N2)));        ax.set_xticklabels(N2, fontsize=10)
ax.set_yticks(range(len(hm_labels))); ax.set_yticklabels(hm_labels, fontsize=9)

# Anotar el F1 en cada celda; texto en blanco si el fondo es muy oscuro o muy claro
for i in range(len(hm_labels)):
    for j in range(len(N2)):
        v = matrix[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                color="white" if v < 0.3 or v > 0.85 else "black")

plt.colorbar(im, ax=ax, label="F1")
ax.set_title("F1 por etiqueta N2 y experimento", pad=12)
plt.tight_layout()
plt.savefig("../results/fig_heatmap_n2.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Histograma de confianza por experimento.
# Permite ver si el modelo sobreestima su certeza (picos en 0.95-1.0)
# o si hay dispersión real que pueda usarse como señal de revisión manual.
fig, axes = plt.subplots(1, len(experimentos_cfg), figsize=(16, 3), sharey=True)
for ax, (nombre, modelo, config, path) in zip(axes, experimentos_cfg):
    conf = pd.read_csv(path)["confidence"].dropna()
    ax.hist(conf, bins=20, range=(0, 1), color="#4C72B0", edgecolor="white")
    ax.axvline(conf.mean(), color="#DD8452", linestyle="--", linewidth=1.5,
               label=f"μ={conf.mean():.2f}")
    ax.set_title(nombre.split("-")[1].strip() if "-" in nombre else nombre, fontsize=9)
    ax.set_xlabel("Confianza", fontsize=8)
    ax.legend(fontsize=8)
axes[0].set_ylabel("Frecuencia")
plt.suptitle("Distribución de confianza por experimento", y=1.03)
plt.tight_layout()
plt.savefig("../results/fig_calibracion_histograma.png", dpi=150, bbox_inches="tight")
plt.show()


---

##  13. Calibración de confianza

¿Es `confidence` un predictor real de calidad o un artefacto del modelo?

In [ ]:
# Analizar si la confianza del LLM predice su exactitud real.
# Un modelo bien calibrado debería tener mayor exact match cuando reporta confianza alta.
# Se considera "predictor útil" si la diferencia de exactitud entre
# confianza >=0.95 y <0.95 es mayor de 0.1 puntos.
calibracion_cfg = [
    ("Qwen 3.5 9B - Baseline",  "../results/exp1_baseline_qwen9b.csv"),
    ("Qwen 3.5 9B - Prompt v2", "../results/exp2_promptv2_qwen9b.csv"),
    ("Qwen 3.5 9B - +N1",       "../results/exp3_promptv2_n1_qwen9b.csv"),
    ("Qwen 3.5 9B - Few-shot",  "../results/exp4_fewshot_qwen9b.csv"),
    ("Gemma 4 4B - Prompt v2",  "../results/exp5_promptv2_gemma4b.csv"),
]
print(f"{'Experimento':<30} {'Media':>7} {'Min':>7} {'>=0.95':>8} {'Predictor?':>12}")
print("─" * 70)
for nombre, path in calibracion_cfg:
    df_r = pd.read_csv(path)
    df_e = df_anotado[["id","is_relevant_gt","procedures_gt","description"]].merge(
        df_r[["description","is_relevant_pred","procedures_pred","confidence"]], on="description", how="left"
    )
    # Excluir filas con error (confidence=NaN)
    df_e = df_e[df_e["confidence"].notna()]
    df_e["exact"] = df_e.apply(
        lambda r: parse_labels(r["procedures_gt"]) == parse_labels(r["procedures_pred"]), axis=1
    )
    conf     = df_e["confidence"]
    alta_acc = df_e[conf >= 0.95]["exact"].mean()
    baja     = df_e[conf < 0.95]
    # Solo concluir si hay suficientes muestras de baja confianza para comparar
    if len(baja) >= 3:
        predictor = "✅ Sí" if (alta_acc - baja["exact"].mean()) > 0.1 else "❌ No"
    else:
        predictor = "- insuf."
    print(f"{nombre:<30} {conf.mean():>7.3f} {conf.min():>7.3f} {(conf>=0.95).mean():>7.1%}  {predictor:>12}")
print()
print("Conclusión: solo Gemma 4B expresa incertidumbre real (min=0.100).")
print("Qwen 9B es sobreconfiante - confidence no sirve como filtro de revisión.")


---

##  13. Calibración de confianza

¿Es `confidence` un predictor real de calidad o un artefacto del modelo?

---

##  14. Conclusiones y trabajo futuro

### Hallazgos principales

**1. El prompt engineering es el factor de mayor impacto.**
El salto de Baseline a Prompt v2 supone +18 puntos de Macro-F1 (0.777 → 0.957) sin cambiar el modelo ni la arquitectura. La redefinición del dominio resolvió por sí sola los fallos sistemáticos en IIA, IAE y AAI no energéticos.

**2. El few-shot quirúrgico aporta mejoras modestas pero consistentes.**
Tres ejemplos seleccionados sobre los casos de error sistemático añaden +1.4 puntos sobre Prompt v2 (0.957 → 0.971). El principio "few-shot dirigido a errores conocidos" es más eficiente que few-shot exhaustivo.

**3. El contexto N1 mejora la detección de relevancia, no la clasificación N2.**
La configuración +N1 mejora is_relevant F1 de 0.973 a 0.986 pero no mejora Macro-F1 (0.957 vs 0.956). El LLM infiere correctamente la forma jurídica desde el texto.

**4. Gemma 4B es competitivo con Qwen 9B a menor tamaño.**
Macro-F1 = 0.962 vs 0.957 - diferencia no significativa con n=100. Ventaja del Gemma: mejor calibración de confianza (min=0.100 vs 0.950).

**5. AAI es el procedimiento más difícil.** Máximo F1 = 0.889 en todos los experimentos.

**6. La confianza del Qwen 9B no es predictor fiable.** 99% de registros con confidence ≥ 0.95 independientemente del acierto.

### Limitaciones

1. **Ground truth Opción A**: 100 registros anotados sin revisión cruzada.
2. **Modelo local limitado**: Qwen 3.5 9B con 16GB RAM. Sin acceso a modelos frontera completos.
3. **N3 insuficientemente evaluado**: solo `fotovoltaica` y `línea_eléctrica` con volumen suficiente.

### Trabajo futuro - v2

| Tarea | Ref |
|-------|-----|
| Ampliar a 18 familias y 92 procedimientos N2 completo | - |
| Ground truth riguroso ≥500 registros con revisión cruzada | DEC-006 |
| Nuevas tecnologías N3: `industria_ippc`, `infraestructura_hidrica`, `ordenacion_territorial`, `turismo_edificacion` | DEC-014–017 |
| Mejora `línea_eléctrica` en prompt: sinónimos "instalación eléctrica", "centro de transformación" | DEC-021 |
| N1 para RRHH sin tipo explícito: LLM zero-shot | DEC-011 |
| Confirmar scope is_relevant: energético estricto vs ambiental amplio | PENDIENTE-002 |
| Evaluación con modelos frontera (Gemini 2.5 Pro) | - |